# 用于测试动态环境对训练完成的智能体的影响

## beta的动态变化

In [7]:
import numpy as np

from ppo_discrete_gpu import PPO_discrete_gpu
from meta_env_tensor_vector import EpidemicModelTensorVector
import pandas as pd
import warnings
from train import add_noise_to_state
from dynamic.env_change_rules.beta_change import *
import torch

warnings.filterwarnings("ignore")

from config import args
from config import experiment_scenario

print(args.device_name)

cuda:0


In [3]:
# from train_gpu import evaluate_policy

draw_figure = False
print_output = True

def my_test_no_train_beta_change(args, beta_matrix = None):
    model_idx = args.model_idx
    eval_env = EpidemicModelTensorVector(args, env_count=env_count)
    if beta_matrix is not None:
        eval_env.adjust_beta_matrix(beta_matrix)
    args.zone_num = eval_env.ZONE_NUM
    agent = PPO_discrete_gpu(args)
    agent.load(model_idx)
    if print_output:
        print("模型文件加载位置：", agent.directory)

    for i in range(1):
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = evaluate_policy(args, eval_env, agent,
                                                                                   args.use_state_norm)
        if draw_figure:
            eval_env.render()
        daily_new_I = eval_env.daily_new_I.cpu().numpy()
        if print_output:
            print("总感染人数：", daily_new_I.mean(axis=0).sum())
            print(
                'experiment:%d\t model:%d\t reward:%.2f\t overload_ratio:%.2f%%\t intensity:%.3f\t sdo:%.3f tdo:%.3f fdo:%.3f ado:%.3f' %
                (args.experiment_idx, args.model_idx,
                 e_r,
                 e_overload * 100,
                 e_intensity / eval_env.period / eval_env.ZONE_NUM,
                 e_sdo / eval_env.period,
                 e_tdo / eval_env.ZONE_NUM,
                 e_fdo / eval_env.period,
                 e_ado / eval_env.period)
            )
        
    return e_r, e_overload, e_intensity / eval_env.period / eval_env.ZONE_NUM, e_sdo, e_fdo, e_tdo, e_ado

def my_test_train_beta_change(args, beta_matrix):
    model_idx = args.model_idx
    eval_env = EpidemicModelTensorVector(args, env_count=env_count)
    eval_env.adjust_beta_matrix(beta_matrix)
    eval_env.state_contain_beta = True
    args.zone_num = eval_env.ZONE_NUM
    agent = PPO_discrete_gpu(args)
    print("模型文件加载位置：", agent.directory)
    if print_output:
        print("模型文件加载位置：", agent.directory)
    agent.load(model_idx)
    

    for i in range(1):
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = evaluate_policy(args, eval_env, agent,
                                                                                   args.use_state_norm)
        if draw_figure:
            eval_env.render()
        daily_new_I = eval_env.daily_new_I.cpu().numpy()
        if print_output:
            print("总感染人数：", daily_new_I.mean(axis=0).sum())
            print(
                'experiment:%d\t model:%d\t reward:%.2f\t overload_ratio:%.2f%%\t intensity:%.3f\t sdo:%.3f tdo:%.3f fdo:%.3f ado:%.3f' %
                (args.experiment_idx, args.model_idx,
                 e_r,
                 e_overload * 100,
                 e_intensity / eval_env.period / eval_env.ZONE_NUM,
                 e_sdo / eval_env.period,
                 e_tdo / eval_env.ZONE_NUM,
                 e_fdo / eval_env.period,
                 e_ado / eval_env.period)
            )
        
    return e_r, e_overload, e_intensity / eval_env.period / eval_env.ZONE_NUM, e_sdo, e_fdo, e_tdo, e_ado
        
        
args.device_name = "cpu"
env_count = 1
device = torch.device(args.device_name)
beta_matrix = np.full((env_count, args.ODE_period, args.zone_num), args.ODE_beta)
beta_dynamic_std = [0.1, 0.2, 0.3, 0.4]



In [5]:
# 1.时序变化的beta
print_output = False
draw_figure = False
args.train_beta_change_rule = "none"
beta_daily = np.full((args.ODE_period, 1), args.ODE_beta)
for std in beta_dynamic_std:
    print("beta变化方差为：", std)
    # 重复实验3次
    e_rs, e_overloads, e_intensities = [], [], []
    e_rs_train_change, e_overloads_train_change, e_intensities_train_change = [], [], []
    for _ in range(30):
        # 生成均匀分布噪声，范围从-std到std
        uniform_noise = np.random.uniform(0, std, beta_daily.shape)
        noisy_beta_daily = beta_daily + uniform_noise
        # 将beta_daily 扩展到beta_matrix的形状
        noisy_beta_matrix = noisy_beta_daily.repeat(args.zone_num, axis=1)
        noisy_beta_matrix = torch.from_numpy(noisy_beta_matrix).to(device).float()
        noisy_beta_matrix = noisy_beta_matrix.unsqueeze(0).expand(env_count, -1, -1)
        
        args.train_beta_change_rule = "none"
        args.test_beta_change_rule = "none"
        args.model_idx = 100
        args.state_dim = 7
        args.max_train_steps = int(1.2e6)
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = my_test_no_train_beta_change(args, noisy_beta_matrix)
        e_rs.append(e_r)
        e_overloads.append(e_overload)
        e_intensities.append(e_intensity)
        
        # args.model_idx = 200
        # args.train_beta_change_rule = "temporal"
        # args.test_beta_change_rule = "none"
        # args.timenow = "_2024-09-04_13-35_"
        # args.max_train_steps = int(2.4e6)
        args.model_idx = 190
        args.train_beta_change_rule = "spatial"
        args.test_beta_change_rule = "none"
        args.timenow = "_2024-09-04_00-57_"
        args.state_dim = 7 * 2
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = my_test_train_beta_change(args, noisy_beta_matrix)
        e_rs_train_change.append(e_r)
        e_overloads_train_change.append(e_overload)
        e_intensities_train_change.append(e_intensity)
        
    print("平均值：", np.mean(e_rs), np.mean(e_overloads), np.mean(e_intensities), "*" * 80)
    print("beta变化下训练后平均值：", np.mean(e_rs_train_change), np.mean(e_overloads_train_change), np.mean(e_intensities_train_change))

beta变化方差为： 0.1
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加

In [15]:
# 2.空间变化的beta
# np.random.seed(42)
draw_figure = False
print_output = False
args.train_beta_change_rule = "none"
# beta_spatial = np.array([args.ODE_beta] * args.zone_num)
beta_spatial = np.full((1, args.zone_num), args.ODE_beta)
for std in beta_dynamic_std:
    print("beta变化方差为：", std, "=" * 80)
    # 重复实验3次
    e_rs, e_overloads, e_intensities = [], [], []
    # 
    e_rs_train_change, e_overloads_train_change, e_intensities_train_change = [], [], []
    for _ in range(20):
        # 生成均匀分布噪声，范围从-std到std
        uniform_noise = np.random.uniform(0, std, beta_spatial.shape)
        noisy_beta_spatial = beta_spatial + uniform_noise
        # 将beta_spatial 扩展到beta_matrix的形状        
        noise_beta_matrix = noisy_beta_spatial.repeat(args.ODE_period, axis=0)
        noisy_beta_matrix = torch.from_numpy(noise_beta_matrix).to(device).float()
        noisy_beta_matrix = noisy_beta_matrix.unsqueeze(0).expand(env_count, -1, -1)    
        
        args.train_beta_change_rule = "none"
        args.test_beta_change_rule = "none"
        args.model_idx = 100
        args.state_dim = 7
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = my_test_no_train_beta_change(args, noisy_beta_matrix)
        e_rs.append(e_r)
        e_overloads.append(e_overload)
        e_intensities.append(e_intensity)
        
        args.model_idx = 190
        args.train_beta_change_rule = "spatial"
        args.test_beta_change_rule = "none"
        args.timenow = "_2024-09-04_00-57_"
        args.state_dim = 7 * 2
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = my_test_train_beta_change(args, noisy_beta_matrix)
        e_rs_train_change.append(e_r)
        e_overloads_train_change.append(e_overload)
        e_intensities_train_change.append(e_intensity)
    print("平均值：", np.mean(e_rs), np.mean(e_overloads), np.mean(e_intensities), "*" * 80)
    print("beta变化下训练后平均值：", np.mean(e_rs_train_change), np.mean(e_overloads_train_change), np.mean(e_intensities_train_change))


beta变化方差为： 0.1 ================================================================================
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gp

KeyboardInterrupt: 

In [17]:
# 3.时空异质性的beta
# 对beta_matrix 每个元素添加高斯噪声
# for std in beta_dynamic_std:
#     print("beta变化方差为：", std)
#     noisy_beta_matrix = beta_matrix + np.random.normal(0, std, beta_matrix.shape)
#     noisy_beta_matrix = torch.from_numpy(noisy_beta_matrix).to(device).float()
#     my_test_no_train_beta_change(args, noisy_beta_matrix)

draw_figure = False
# 对beta_matrix每个元素添加不同标准差的均匀分布噪声
for std in beta_dynamic_std:
    print("beta变化方差为：", std)
    e_rs, e_overloads, e_intensities = [], [], []
    e_rs_train_change, e_overloads_train_change, e_intensities_train_change = [], [], []
    # 重复实验3次
    for _ in range(10):
        # 生成均匀分布噪声，范围从-std到std
        uniform_noise = np.random.uniform(0, std, beta_matrix.shape)
        noisy_beta_matrix = beta_matrix + uniform_noise
        
        noisy_beta_matrix = torch.from_numpy(noisy_beta_matrix).to(device).float()
        noisy_beta_matrix = noisy_beta_matrix.expand(env_count, -1, -1)
        
        
        args.train_beta_change_rule = "none"
        args.test_beta_change_rule = "none"
        args.model_idx = 100
        args.state_dim = 7
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = my_test_no_train_beta_change(args, noisy_beta_matrix)
        e_rs.append(e_r)
        e_overloads.append(e_overload)
        e_intensities.append(e_intensity)
        
        args.model_idx = 190
        args.train_beta_change_rule = "spatial"
        args.test_beta_change_rule = "none"
        args.timenow = "_2024-09-04_00-57_"
        args.state_dim = 7 * 2
        e_r, e_overload, e_intensity, e_sdo, e_fdo, e_tdo, e_ado = my_test_train_beta_change(args, noisy_beta_matrix)
        e_rs_train_change.append(e_r)
        e_overloads_train_change.append(e_overload)
        e_intensities_train_change.append(e_intensity)
    print("平均值：", np.mean(e_rs), np.mean(e_overloads), np.mean(e_intensities), "*" * 80)
    print("beta变化下训练后平均值：", np.mean(e_rs_train_change), np.mean(e_overloads_train_change), np.mean(e_intensities_train_change))

beta变化方差为： 0.1
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
使用了beta变化
使用了beta变化
模型文件加载位置： .\model\gpu\dynamic_env\sz_4\betachange=spatial_2024-09-04_00-57_
平均值： -67.31177215576172 0

### 动态环境训练出的 actor 是否表现更好

In [19]:
args.city = "sz"
args.R0 = 'high'
args.experiment_idx = 4

args.model_idx = 100
args.max_train_steps = int(1.2e6)
args.beta_change_rule = BetaChangeRule.NONE



# 1.测试环境设置为环境动态

# 1.1.无变化的beta
# args.test_beta_change_rule = BetaChangeRule.NONE

# 1.2.时序变化的beta
args.test_beta_change_rule = BetaChangeRule.TEMPORAL
beta_temporal = [args.ODE_beta] * args.ODE_period
change_period = np.arange(10, 31, 1)
for i in change_period:
    beta_temporal[i] = 1.3 * args.ODE_beta
args.beta_temporal = beta_temporal

# 2.选择要加载的模型
# 2.1.无动态环境下训练完成的智能体
args.beta_change_rule = BetaChangeRule.NONE
output = my_test_no_train_beta_change(args)
print(output)

# 2.2.动态环境下训练完成的智能体 1.15
args.beta_change_rule = BetaChangeRule.TEMPORAL
args.timenow = "_2024-08-29_22-22_"
output = my_test_no_train_beta_change(args)
print(output)

# 2.2.动态环境下训练完成的智能体 1.2
args.beta_change_rule = BetaChangeRule.TEMPORAL
args.timenow = "_2024-08-29_21-05_"
output = my_test_no_train_beta_change(args)
print(output)



总新增感染人数：17045797.955738854，现存感染最大峰值：5320380.084515833
{'reward': [-205.0998684922376], 'IOR': [0.33009502112895833], 'ACI': [0.41441441441441446], 'ATO': [5.0675675675675675], 'ASO_adj': [0.39950161102352505], 'ASO_mob': [0.33558667041670015], 'ASO_adm': [0.4367830085652779]}
总新增感染人数：16959564.04974347，现存感染最大峰值：4446248.944053961
{'reward': [-106.588018866881], 'IOR': [0.11156223601349025], 'ACI': [0.5529279279279279], 'ATO': [5.918918918918919], 'ASO_adj': [0.41587780900975935], 'ASO_mob': [0.34979541936649966], 'ASO_adm': [0.480599805854666]}
总新增感染人数：16931354.48438543，现存感染最大峰值：4265655.623149168
{'reward': [-95.43667578845935], 'IOR': [0.06641390578729194], 'ACI': [0.6341216216216216], 'ATO': [6.094594594594595], 'ASO_adj': [0.46196026246572136], 'ASO_mob': [0.3940935749421841], 'ASO_adm': [0.5290852696661384]}


总新增感染人数：16942803.844907865，现存感染最大峰值：4378879.695504723
{'reward': [-88.0018536681768], 'IOR': [0.09471992387618078], 'ACI': [0.43873873873873875], 'ATO': [4.851351351351352], 'ASO_adj': [0.3949862639828775], 'ASO_mob': [0.32758313192304944], 'ASO_adm': [0.43351562179936975]}
总新增感染人数：16981498.373697177，现存感染最大峰值：4692031.686076831
{'reward': [-125.29109283812039], 'IOR': [0.17300792151920777], 'ACI': [0.42713963963963963], 'ATO': [4.9324324324324325], 'ASO_adj': [0.395226489133692], 'ASO_mob': [0.32828420877993936], 'ASO_adm': [0.4348325998693288]}
总新增感染人数：17045797.955738854，现存感染最大峰值：5320380.084515833
{'reward': [-205.0998684922376], 'IOR': [0.33009502112895833], 'ACI': [0.41441441441441446], 'ATO': [5.0675675675675675], 'ASO_adj': [0.39950161102352505], 'ASO_mob': [0.33558667041670015], 'ASO_adm': [0.4367830085652779]}
